# E02 — Cantilever Snap-Fit Design
*Exam tool — simplified 3-part structure. For full analysis see `03_snap_fit.ipynb`.*

---

## Part 1 — Theory Recap

### Root Strain
$$\varepsilon_{\max} = \frac{1.5 \cdot h \cdot Y}{L^2 \cdot Q}$$
where $h$ = root thickness, $Y$ = required deflection (undercut), $L$ = beam length, $Q$ = wall compliance factor ($Q = 1$ for rigid wall).

### Allowable Strain (Repeated Assembly)
$$\varepsilon_{\text{allow}} = 0.6 \times \varepsilon_{\text{yield}}$$
The factor 0.6 accounts for cyclic fatigue reduction. Use $\varepsilon_{\text{yield}}$ from the material datasheet.

### Deflection Force (using Secant Modulus)
$$F_d = \frac{b \cdot h^2 \cdot E_s \cdot \varepsilon_{\max}}{6 \cdot L}$$
**$E_s$ is the secant modulus** at the operating strain — NOT the instantaneous modulus. For exam problems, $E_s$ is either given directly or approximated as the initial flexural modulus.

### Assembly and Disassembly Forces
$$F_a = F_d \cdot \frac{\mu + \tan\alpha}{1 - \mu \tan\alpha} \qquad
  F_{\text{dis}} = F_d \cdot \frac{\mu + \tan\beta}{1 - \mu \tan\beta}$$
where $\alpha$ = lead-in angle, $\beta$ = return (retention) angle.

### Inseparable Condition
Joint becomes permanently locked when the denominator $\to 0$:
$$\beta_{\text{lock}} = \arctan\!\left(\frac{1}{\mu}\right)$$
For POM ($\mu = 0.20$): $\beta_{\text{lock}} = 78.7°$

### Beam Theory Validity Check
$$Y/L \leq 0.20 \quad \text{(classical Euler–Bernoulli beam valid)}$$

### Parameter Table
| Symbol | Description | Unit |
|--------|-------------|------|
| $h$ | Beam thickness at root | m |
| $b$ | Beam width | m |
| $L$ | Effective beam length | m |
| $Y$ | Required deflection (undercut height) | m |
| $Q$ | Wall compliance factor (1 = rigid) | — |
| $E_s$ | Secant flexural modulus | Pa |
| $\varepsilon_{\text{yield}}$ | Material yield strain | — |
| $\mu$ | Friction coefficient (polymer on steel) | — |
| $\alpha$ | Lead-in angle | ° |
| $\beta$ | Return angle | ° |

---

## ⚠ Common Exam Pitfalls
1. **h vs b confusion** — deflection force uses $h^2$ (thickness squared), not $b^2$.
2. **Wrong modulus** — use the **secant modulus** $E_s$ at operating strain; if not given, use flexural modulus.
3. **Q = 1 default** — if wall compliance is not stated, assume $Q = 1$ (rigid wall).
4. **Return angle direction** — $\beta = 90°$ means perpendicular shoulder = nearly permanent (very high $F_{\text{dis}}$).

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Problem Inputs  (edit values here)
# ═══════════════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '..')
import numpy as np

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import FRICTION_COEFFICIENTS

# ── Geometry ─────────────────────────────────────────────────────────────────
h_root  = Q_(2.5, 'mm')   # beam thickness at root
b_width = Q_(8.0, 'mm')   # beam width
L_beam  = Q_(25.0, 'mm')  # effective beam length
Y_defl  = Q_(2.0, 'mm')   # required deflection (undercut height)
Q_wall  = 1.0              # wall compliance factor (1 = rigid wall)

# ── Angles ───────────────────────────────────────────────────────────────────
alpha_deg = 30.0   # lead-in angle [°]
beta_deg  = 60.0   # return (retention) angle [°]

# ── Material — POM ───────────────────────────────────────────────────────────
E_secant  = Q_(2800.0, 'MPa')   # secant flexural modulus (use E_instant if secant unknown)
eps_yield = 0.06                 # material yield strain (POM: ~6%)
mu = FRICTION_COEFFICIENTS['POM_steel']   # 0.20

print(f'Friction coefficient POM/steel: mu = {mu}')


Friction coefficient POM/steel: mu = 0.2


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Functions
# ═══════════════════════════════════════════════════════════════════════════════

def root_strain(h_m, Y_m, L_m, Q=1.0):
    """Maximum root strain ε = 1.5·h·Y / (L²·Q). Returns dimensionless strain."""
    return 1.5 * h_m * Y_m / (L_m**2 * Q)


def deflection_force(b_m, h_m, E_Pa, eps_max, L_m):
    """Beam deflection force Fd = b·h²·E·ε / (6·L). Returns force in N."""
    return (b_m * h_m**2 * E_Pa * eps_max) / (6.0 * L_m)


def mating_force(Fd_N, mu, angle_deg):
    """Assembly or disassembly force. Returns N (or inf if joint is inseparable)."""
    tan_a = np.tan(np.radians(angle_deg))
    denom = 1.0 - mu * tan_a
    if abs(denom) < 1e-6:
        print(f'  WARNING: joint inseparable at angle={angle_deg:.1f}° (denominator → 0)')
        return float('inf')
    return Fd_N * (mu + tan_a) / denom


print('Functions defined.')


Functions defined.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Execution
# ═══════════════════════════════════════════════════════════════════════════════

h_m = strip_units(h_root.to('m'))
b_m = strip_units(b_width.to('m'))
L_m = strip_units(L_beam.to('m'))
Y_m = strip_units(Y_defl.to('m'))
E_Pa = strip_units(E_secant.to('Pa'))

# --- Step 1: Geometry check --------------------------------------------------
YL_ratio = Y_m / L_m
print('--- Step 1: Geometry Check ---')
print(f"  {'h (root thickness)':<28}: {h_m*1e3:.2f} mm")
print(f"  {'b (width)':<28}: {b_m*1e3:.2f} mm")
print(f"  {'L (beam length)':<28}: {L_m*1e3:.2f} mm")
print(f"  {'Y (required deflection)':<28}: {Y_m*1e3:.2f} mm")
print(f"  {'Y/L ratio':<28}: {YL_ratio:.3f}  {'(OK: classical beam valid)' if YL_ratio <= 0.20 else '(WARN: non-linear effects)'}")

# --- Step 2: Strain ----------------------------------------------------------
eps_max   = root_strain(h_m, Y_m, L_m, Q_wall)
eps_allow = 0.6 * eps_yield
print()
print('--- Step 2: Strain ---')
print(f"  {'ε_max (root strain)':<28}: {eps_max*100:.3f} %")
print(f"  {'ε_yield':<28}: {eps_yield*100:.1f} %")
print(f"  {'ε_allow (0.6 × ε_yield)':<28}: {eps_allow*100:.3f} %")
print(f"  {'Strain margin':<28}: {(eps_allow - eps_max)*100:+.3f} %")

# --- Step 3: Deflection force ------------------------------------------------
Fd = deflection_force(b_m, h_m, E_Pa, eps_max, L_m)
print()
print('--- Step 3: Deflection Force ---')
print(f"  {'E_secant [MPa]':<28}: {E_Pa/1e6:.0f}")
print(f"  {'Fd (deflection force)':<28}: {Fd:.2f} N")

# --- Step 4: Assembly and disassembly forces ---------------------------------
Fa   = mating_force(Fd, mu, alpha_deg)
Fdis = mating_force(Fd, mu, beta_deg)
beta_lock = np.degrees(np.arctan(1.0 / mu))
print()
print('--- Step 4: Assembly / Disassembly Forces ---')
print(f"  {'μ (friction coefficient)':<28}: {mu}")
print(f"  {'α (lead-in angle)':<28}: {alpha_deg}°")
print(f"  {'β (return angle)':<28}: {beta_deg}°")
print(f"  {'F_assembly':<28}: {Fa:.2f} N")
if Fdis == float('inf'):
    print(f"  {'F_disassembly':<28}: INFINITE (permanent snap-fit)")
else:
    print(f"  {'F_disassembly':<28}: {Fdis:.2f} N")
print(f"  {'β_lock (inseparable at)':<28}: {beta_lock:.1f}°")


--- Step 1: Geometry Check ---
  h (root thickness)          : 2.50 mm
  b (width)                   : 8.00 mm
  L (beam length)             : 25.00 mm
  Y (required deflection)     : 2.00 mm
  Y/L ratio                   : 0.080  (OK: classical beam valid)

--- Step 2: Strain ---
  ε_max (root strain)         : 1.200 %
  ε_yield                     : 6.0 %
  ε_allow (0.6 × ε_yield)     : 3.600 %
  Strain margin               : +2.400 %

--- Step 3: Deflection Force ---
  E_secant [MPa]              : 2800
  Fd (deflection force)       : 11.20 N

--- Step 4: Assembly / Disassembly Forces ---
  μ (friction coefficient)    : 0.2
  α (lead-in angle)           : 30.0°
  β (return angle)            : 60.0°
  F_assembly                  : 9.84 N
  F_disassembly               : 33.11 N
  β_lock (inseparable at)     : 78.7°


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Validation
# ═══════════════════════════════════════════════════════════════════════════════

pass_strain = eps_max <= eps_allow
pass_beam   = YL_ratio <= 0.20
overall     = pass_strain and pass_beam

print('--- VALIDATION ---')
print(f"  Strain check  : {eps_max*100:.3f}%  <=  {eps_allow*100:.3f}%"
      f"  |  {'PASS' if pass_strain else 'FAIL'}")
if not pass_beam:
    print(f"  Beam theory   : Y/L={YL_ratio:.3f}  >  0.20  |  WARN (non-linear; result approximate)")
else:
    print(f"  Beam theory   : Y/L={YL_ratio:.3f}  <=  0.20  |  PASS")
print(f"  Snap type     : {'PERMANENT (inseparable)' if beta_deg >= beta_lock else 'DETACHABLE'}  (β={beta_deg}°, β_lock={beta_lock:.1f}°)")
print(f"  OVERALL       : {'PASS' if overall else 'FAIL'}")


--- VALIDATION ---
  Strain check  : 1.200%  <=  3.600%  |  PASS
  Beam theory   : Y/L=0.080  <=  0.20  |  PASS
  Snap type     : DETACHABLE  (β=60.0°, β_lock=78.7°)
  OVERALL       : PASS
